In [1]:
#%pip install -qU langchain langchain_openai langchain-core langgraph

In [15]:
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage
import os
import gradio as gr
import random

In [5]:
with open("/content/openai_key.txt") as archivo:
  apikey = archivo.read()
os.environ["OPENAI_API_KEY"] = apikey

In [43]:
# Diccionario 0-10 en quechua (dialecto quechua sureño básico)
NUMS_Q = ["ch'uñu", "huk", "iskay", "kimsa", "tawa",
          "pichqa", "suqta", "qanchis", "pusaq", "isqon", "chunka"]

def num_to_q(n:int) -> str:
    return NUMS_Q[n] if 0 <= n <= 10 else str(n)

# Traducción de números a quechua
def num_to_q(n):
    num_q = {
        0: "ch’usaq", 1: "huk", 2: "iskay", 3: "kimsa", 4: "tawa",
        5: "pichqa", 6: "soqta", 7: "qanchis", 8: "pusaq", 9: "isqun", 10: "chunka"
    }
    return num_q.get(n, str(n))

# 🔹 Tool 1: Generar problema
@tool
def generar_problema(_: str = "") -> str:
    """Crea un problema de suma o resta (≤10) en quechua para primer grado."""
    while True:
        a, b = random.randint(0, 10), random.randint(0, 10)
        op = random.choice(["+", "-"])
        if op == "-" and b > a:
            a, b = b, a  # Evitar resultados negativos
        resultado = a + b if op == "+" else a - b
        if 0 <= resultado <= 10:
            problema = f"¿Imaynata {num_to_q(a)} {op} {num_to_q(b)}?"
            # Guardar internamente los operandos y operador
            global estado_operacion
            estado_operacion = (a, op, b)
            return problema

# 🔹 Tool 2: Evaluar respuesta
@tool
def evaluar_respuesta(respuesta: str) -> str:
    """Evalúa si la respuesta es correcta comparando con la operación previa."""
    try:
        a, op, b = estado_operacion  # Usa el estado global del problema generado
        correcta = a + b if op == "+" else a - b
        if int(respuesta.strip()) == correcta:
            return "ALLIN! ¡Muy bien!"
        else:
            return "SUT'IY – Intenta otra vez."
    except:
        return "No se encontró el problema anterior para evaluar."

# 🔹 Tool 3: Explicar solución
@tool
def explicar_solucion(_: str = "") -> str:
    """Explica paso a paso la operación anterior en quechua y español."""
    try:
        a, op, b = estado_operacion
        res = a + b if op == "+" else a - b
        op_es = "más" if op == "+" else "menos"
        return (
            f"📘 Paso a paso (Quechua): {num_to_q(a)} ({a}) {op} {num_to_q(b)} ({b}) = {res}\n"
            f"📗 Paso a paso (Español): {a} {op_es} {b} es {res}."
        )
    except:
        return "No hay una operación reciente para explicar."

In [44]:
# 3. Prompt de sistema (Quechua + español para contexto)
SYSTEM = """Eres Yachachiq-Bot 📚, tutor de matemáticas para niños de 6-7 años.
Hablas mayormente en quechua sureño con apoyo en español.

Sigue siempre estos pasos:
0) Saluda amigablemente en Quechua y Español.
1) Usa 'generar_problema' para crear un ejercicio.
2) Espera la respuesta del niño, va a ser un n° ejemplo: 7, 8, ..
3) Llama a 'evaluar_respuesta'
4) Si es correcta, celebra en quechua ("Allinlla!") y genera uno nuevo pero si es incorrecta, usa 'explicar_solucion' y ofrece otro intento.

Nota: Vas a interacturar con un niño que habla español
"""



In [45]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "{messages}"),
])

In [46]:
# Crear agente
toolkit = [generar_problema, evaluar_respuesta, explicar_solucion]
model = ChatOpenAI(verbose=True, temperature=0.2)
memory = MemorySaver()

In [47]:
agent = create_react_agent(
    model,
    toolkit,
    checkpointer = memory,
    prompt=prompt
)

In [49]:
thread_id = "colab_thread_004"

In [50]:
def conversar_agente_v2(message, history):
    config = {"configurable": {"thread_id": thread_id}}
    user_msg = HumanMessage(content=message)

    result = agent.invoke({"messages": [user_msg]}, config=config)
    respuesta = result["messages"][-1].content

    # Asegurarse de que devuelve SOLO la lista de mensajes (formato nuevo)
    return history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": respuesta},
    ]

In [51]:
gr.ChatInterface(
    fn=conversar_agente_v2,
    chatbot=gr.Chatbot(type="messages", height=400),
    title="Yachachiq-Bot 📚",
    description="Pregunta sumas o multiplicaciones.",
).launch(share=True, debug=True)

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:321: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'messages', will be used.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://88046038e7fd0777d3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://88046038e7fd0777d3.gradio.live
